In [ ]:
import os
import sys
import time
import pygame
import joblib
import pandas as pd
import numpy as np
from tkinter import filedialog, Tk, messagebox
from datetime import datetime

# --- 0. 환경 설정 및 로그 시스템 ---
os.environ['SDL_VIDEO_CENTERED'] = '1'
def log(message): print(f"[{time.strftime('%H:%M:%S')}] {message}")

# --------------------------------------------------------------------------
# [1단계] 통합 자산 선택 가이드 (PKL & CSV)
# --------------------------------------------------------------------------
def select_hybrid_assets():
    root = Tk(); root.withdraw(); root.attributes("-topmost", True)
    
    # 박사님 요청 가이드 멘트
    messagebox.showinfo("M-Robo 지능 이식 가이드", 
                        "로봇 시뮬레이션을 시작하기 위해 두 가지 자산을 선택합니다.\n\n"
                        "1. 우승 모델(PKL): 지능 대회에서 선발된 최종 모델\n"
                        "2. 마스터 사전(CSV): 중복 제거 및 라벨 교정이 완료된 사전")
    
    log("📂 1단계: 지능 대회 우승 모델(PKL)을 선택하세요...")
    pkl_path = filedialog.askopenfilename(title="1. 우승 모델(PKL) 선택", filetypes=[("Model", "*.pkl")])
    
    csv_path = None
    if pkl_path:
        log("📂 2단계: 최종 정제된 마스터 사전(CSV)을 선택하세요...")
        csv_path = filedialog.askopenfilename(title="2. 마스터 사전(CSV) 선택", filetypes=[("CSV", "*.csv")])
    
    root.destroy()
    return pkl_path, csv_path

target_model, target_dict = select_hybrid_assets()

if not target_model or not target_dict:
    log("❌ 필수 자산 누락으로 시뮬레이터를 종료합니다.")
    sys.exit()

# --------------------------------------------------------------------------
# [2단계] 하이브리드 추론 엔진 (박사님 V5.2 정통 로직 동기화)
# --------------------------------------------------------------------------
class HybridSimulatorEngine:
    def __init__(self, m_path, d_path):
        log("🧠 우승 모델 로드 중...")
        self.model = joblib.load(m_path)
        
        try:
            df_dict = pd.read_csv(d_path, encoding='utf-8-sig')
        except:
            df_dict = pd.read_csv(d_path, encoding='cp949')
        
        # 박사님 데이터 변환 로직 적용
        k_col = '텍스트' if '텍스트' in df_dict.columns else df_dict.columns[0]
        v_col = '정답_및_오답_여부' if '정답_및_오답_여부' in df_dict.columns else df_dict.columns[1]
        t_col = '의도_Target' if '의도_Target' in df_dict.columns else None
        
        # 사전 구축 (공백 제거 및 소문자화)
        self.word_map = dict(zip(df_dict[k_col].astype(str).str.replace(" ","").str.lower(), df_dict[v_col]))
        self.intent_map = dict(zip(df_dict[k_col].astype(str).str.replace(" ","").str.lower(), df_dict[t_col])) if t_col else {}
        
        from sentence_transformers import SentenceTransformer
        log("📥 SBERT 언어 엔진 가동 중...")
        self.embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        log(f"✅ 엔진 동기화 완료 (사전: {len(self.word_map)}건)")

    def analyze(self, text):
        clean = text.replace(" ", "").lower()
        
        # 1순위: 사전 분석 (명사수)
        if clean in self.word_map:
            label = int(float(self.word_map[clean]))
            target = self.intent_map.get(clean, "알 수 없음")
            # 박사님 요청 출력 포맷 (V5.2 일치)
            action_msg = f"'{target}' 실행/차단"
            return label, "🎯 [사전]", 1.0, action_msg, f"🎯 [명사수] 사전 일치"
        
        # 2순위: AI 모델 추론 (우등생/탐정)
        vec = self.embedder.encode([text])
        prob = self.model.predict_proba(vec)[0]
        pred = 0 if prob[0] >= 0.40 else 1
        score = prob[0] if pred == 0 else prob[1]
        
        if score >= 0.85: analogy = "🧐 [우등생] 고확신 추론"
        elif score >= 0.60: analogy = "🤔 [탐정] 맥락 유추"
        else: analogy = "❓ [외국인] 근거 불분명"
        
        return pred, "🧐 [AI모델]", score, "AI 판단 수행", analogy

clf = HybridSimulatorEngine(target_model, target_dict)

# --------------------------------------------------------------------------
# [3단계] Pygame 시각화 인터페이스 설정
# --------------------------------------------------------------------------
pygame.init()
screen = pygame.display.set_mode((800, 650))
pygame.display.set_caption("M-Robo Hybrid Intelligence Simulator V21.8")

def get_korean_font(size):
    fonts = ["malgungothic", "applesdgothicneo", "nanumgothic", "arial"]
    for f in fonts:
        try: return pygame.font.SysFont(f, size)
        except: continue
    return pygame.font.SysFont(None, size)

m_font = get_korean_font(24); i_font = get_korean_font(18)
car_y = 480; status_msg = "명령 대기 중..."; analogy_msg = ""; info_msg = ""
running = True

# --------------------------------------------------------------------------
# [4단계] 실시간 시뮬레이션 및 애니메이션 루프
# --------------------------------------------------------------------------
log("🚀 시뮬레이터 가동. 터미널에 명령을 입력하세요.")

while running:
    screen.fill((245, 245, 245))
    pygame.draw.rect(screen, (40, 40, 40), (300, 0, 200, 650)) # 도로
    
    # 색상 제어
    if "🛑" in status_msg: c_color = (220, 20, 60)
    elif "🟢" in status_msg: c_color = (30, 144, 255)
    else: c_color = (169, 169, 169)
    
    pygame.draw.rect(screen, c_color, (375, car_y, 50, 90), border_radius=15)
    screen.blit(m_font.render(f"로봇 상태: {status_msg}", True, (33, 33, 33)), (30, 40))
    screen.blit(m_font.render(f"지능 분석: {analogy_msg}", True, (0, 102, 204)), (30, 90))
    screen.blit(i_font.render(info_msg, True, (80, 80, 80)), (30, 140))
    pygame.display.flip()

    for event in pygame.event.get():
        if event.type == pygame.QUIT: running = False

    try:
        user_input = input("\n🎤 명령 입력 (q:종료) >> ").strip()
        if user_input.lower() == 'q': break
        
        # 엔진 분석 (메시지 동기화)
        p, src, sc, msg, analogy = clf.analyze(user_input)
        
        status_msg = "🟢 [실행]" if p == 0 else "🔴 [무시]"
        analogy_msg = analogy
        info_msg = f"근거: {src} | 신뢰도: {sc*100:.1f}% | 판단: {msg}"
        
        # 콘솔 출력 (박사님 요청 데이터 포맷 일치 확인)
        print(f"📡 '{user_input}' -> {status_msg} | {info_msg}")

        # 물리 제어 애니메이션
        if p == 0:
            if "멈춰" in msg:
                status_msg = f"🛑 {status_msg} - 정지 확인"
            else:
                for _ in range(40):
                    car_y -= 6
                    if car_y < -100: car_y = 650
                    screen.fill((245,245,245))
                    pygame.draw.rect(screen, (40,40,40), (300, 0, 200, 650))
                    pygame.draw.rect(screen, (30,144,255), (375, car_y, 50, 90), border_radius=15)
                    pygame.display.flip(); time.sleep(0.01); pygame.event.pump()
                    
    except Exception as e: log(f"오류: {e}"); break

pygame.quit()

[16:40:26] 📂 1단계: 지능 대회 우승 모델(PKL)을 선택하세요...
[16:40:31] 📂 2단계: 최종 정제된 마스터 사전(CSV)을 선택하세요...
[16:40:39] 🧠 우승 모델 로드 중...
[16:40:39] 📥 SBERT 언어 엔진 가동 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[16:40:45] ✅ 엔진 동기화 완료 (사전: 6087건)
[16:40:46] 🚀 시뮬레이터 가동. 터미널에 명령을 입력하세요.



🎤 명령 입력 (q:종료) >>  앞으로 가


📡 '앞으로 가' -> 🟢 [실행] | 근거: 🎯 [사전] | 신뢰도: 100.0% | 판단: '앞으로 가' 실행/차단


... (22 outputs left out)


🎤 명령 입력 (q:종료) >>  앞으로 가지마


📡 '앞으로 가지마' -> 🟢 [실행] | 근거: 🧐 [AI모델] | 신뢰도: 61.1% | 판단: AI 판단 수행



🎤 명령 입력 (q:종료) >>  앞으로 가면 절대 안된다.


📡 '앞으로 가면 절대 안된다.' -> 🟢 [실행] | 근거: 🧐 [AI모델] | 신뢰도: 70.3% | 판단: AI 판단 수행
